# 01 — Original policy (the reference loop)

This is the **no-intervention baseline** and the shared skeleton the other
three notebooks plug into. Read this one first: notebooks 02–04 each
describe themselves as "hook A/B/C of the loop defined in 01".

## What "original policy" means

It is a *condition*, not a reference architecture. Each backbone runs
**exactly as its own paper and released checkpoint define it**. Nothing
about the model is standardised, and no weights are touched anywhere in
these notebooks.

What *is* standardised is the loop around the policy:

| this notebook fixes | the policy still owns |
|---|---|
| when the observation is read | image preprocessing (resize, normalise) |
| the order actions reach `env.step` | vision encoder |
| how success is decided | LLM architecture and depth |
| how latency is counted | action tokeniser / de-tokeniser |

The policy is a black box: it only has to expose
`step(image, instruction) -> (T, action_dim)` and `reset()`. That is the
point — if two backbones are scored by different loops, a difference between
them says nothing about the backbones.

## The control loop, and the three places a method can attach

```
                    ┌─────────────────────────────────────────┐
                    │                                         │
              ┌─────▼──────┐                                  │
              │  observe   │   obs = env.step(action)         │
              └─────┬──────┘                                  │
                    │  raw camera frame (H, W, 3) uint8       │
        ╔═══════════▼═══════════╗                             │
        ║  HOOK A                ║  ← 02 foveation            │
        ║  transform the image   ║                             │
        ╚═══════════╤═══════════╝                             │
                    │                                         │
              ┌─────▼──────────────────────────────┐          │
              │  policy.step(image, instruction)   │          │
              │                                    │          │
              │   ├─ preprocess (resize/normalise) │          │
              │   ├─ vision encoder                │          │
              │   ├─ LLM decoder stack ────────────┼──╗       │
              │   └─ action de-tokenise            │  ║       │
              └─────┬──────────────────────────────┘  ║       │
                    │  actions (T, action_dim)        ║       │
        ╔═══════════▼═══════════╗            ╔════════▼═════╗ │
        ║  HOOK B                ║           ║  HOOK C      ║ │
        ║  transform the actions ║           ║  bypass      ║ │
        ║  ← 03 action repeat    ║           ║  layers      ║ │
        ╚═══════════╤═══════════╝            ║  ← 04 depth  ║ │
                    │                        ╚══════════════╝ │
              ┌─────▼──────┐                                  │
              │  env.step  │──────────────────────────────────┘
              └────────────┘
```

| hook | what it touches | when it runs | notebook |
|---|---|---|---|
| **A** | the raw camera frame, **before** the policy's own preprocessing | every control step | `02_fixed_foveation` |
| **B** | the action array the policy returned, **before** `env.step` | every control step | `03_action_repeat` |
| **C** | the decoder-layer modules inside the LLM | once per run (calibrate), then in effect for every forward | `04_fixed_depth_pruning` |

### Why the hook points do not change with the backbone

Every one of these methods is defined at a point in the loop that **exists in
every VLA**, not at a point specific to one architecture:

* **Hook A** is defined on the *environment's* frame. Whatever the policy does
  next — SigLIP patches, a VQ tokeniser, whatever — it starts from that frame.
* **Hook B** is defined on the *action array*. Every policy returns one.
* **Hook C** is defined on a `torch.nn.ModuleList` of decoder layers. Every
  LLM-based VLA has one, though it sits at a different attribute path per
  wrapper (`04` walks candidate paths rather than hard-coding one).

That is what makes the comparison meaningful: if backbone A and backbone B are
hooked at different places, a difference in their results says nothing about
the backbones. So when porting to a new benchmark, **keep the hook points and
change only the env/policy adapters.**

## Everything benchmark-specific lives in one object

Simulators disagree about almost every detail of the interface, and the
disagreements are silent or fatal rather than informative:

| | LIBERO (robosuite) | SimplerEnv (gymnasium) | CALVIN |
|---|---|---|---|
| `reset()` returns | `obs` | `(obs, info)` | `obs` |
| `step()` returns | 4-tuple `(obs, r, done, info)` | **5-tuple** `(obs, r, terminated, truncated, info)` | 4-tuple |
| success reported via | `done` | `info["success"]` | completed subtasks in a sequence |
| image lives at | `obs["agentview_image"]` | `obs["image"][cam]["rgb"]` | `obs["rgb_obs"]["rgb_static"]` |
| settle period at reset | 10 no-op steps | none | none |

A loop that hard-codes any row of that table is not portable, and unpacking
a 5-tuple into four names raises `ValueError` the moment it meets gymnasium.

So all of it goes into an **`EnvAdapter`**. The loop below touches the
simulator only through that object: porting to a new benchmark means writing
one adapter — not editing the loop, and never editing the hooks.

The defaults assume **nothing**: no settle period, success from the
termination flag or from `info["success"]` when the env provides one.
LIBERO's conventions are supplied by a LIBERO adapter, as an example rather
than as the baseline.

In [ ]:
import time
import numpy as np


def _unpack_reset(out):
    """gymnasium returns (obs, info); classic gym returns obs."""
    if isinstance(out, tuple) and len(out) == 2 and isinstance(out[1], dict):
        return out[0], out[1]
    return out, {}


def _unpack_step(out):
    """-> (obs, reward, terminated, truncated, info) for either API."""
    if not isinstance(out, tuple):
        raise TypeError(f"env.step must return a tuple, got {type(out)}")
    if len(out) == 5:
        obs, reward, terminated, truncated, info = out
        return obs, reward, bool(terminated), bool(truncated), info
    if len(out) == 4:
        obs, reward, done, info = out
        return obs, reward, bool(done), False, info
    raise ValueError(
        f"env.step returned {len(out)} values; expected 4 (gym) or 5 "
        f"(gymnasium). Wrap the env or pass a custom adapter.")


def default_is_success(obs, terminated, info):
    """Prefer an explicit success flag; fall back to termination.

    Stated rather than assumed, because the two disagree: a gymnasium env
    can terminate on a time limit with success False, while LIBERO signals
    success through `done` and puts nothing in info.
    """
    if isinstance(info, dict) and "success" in info:
        return bool(info["success"])
    return bool(terminated)


def default_grasped(info):
    """Was the source object EVER grasped during this episode?

    ManiSkill2_real2sim keeps this as a cumulative flag in
    `info["episode_stats"]`, so reading the last step's info is enough.
    It separates "never touched the object" from "grasped it and lost
    it", which is what the failure-type analysis is made of, and it is
    independent of whether the episode ultimately succeeded.

    Returns None rather than False when the env does not report it, so
    "not grasped" and "not measured" stay distinguishable in the file.
    """
    if not isinstance(info, dict):
        return None
    stats = info.get("episode_stats")
    if not isinstance(stats, dict) or "is_src_obj_grasped" not in stats:
        return None
    return bool(stats["is_src_obj_grasped"])


def to_jsonable(value):
    """numpy scalars and arrays -> plain Python.

    An env's info dict is full of numpy bools, and json.dump refuses
    them. Without this the run finishes and then dies on the last line,
    when writing the file -- the most expensive place to fail.
    """
    if isinstance(value, dict):
        return {str(k): to_jsonable(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [to_jsonable(v) for v in value]
    if isinstance(value, np.ndarray):
        return to_jsonable(value.tolist())
    if isinstance(value, np.generic):
        return value.item()
    if value is None or isinstance(value, (str, int, float, bool)):
        return value
    return repr(value)


class EnvAdapter:
    """Wraps a simulator so the loop never sees benchmark-specific details.

    Only `get_image` is mandatory -- there is no defensible default for
    where the camera frame lives, and guessing one would silently read the
    wrong camera rather than fail.
    """

    def __init__(self, env, get_image, is_success=default_is_success,
                 grasped=default_grasped,
                 noop_action=None, settle_steps=0, max_steps=220):
        self.env = env
        self.get_image = get_image
        self.is_success = is_success
        self.grasped = grasped
        self.noop_action = noop_action
        self.settle_steps = int(settle_steps)
        self.max_steps = int(max_steps)
        if self.settle_steps and self.noop_action is None:
            raise ValueError("settle_steps > 0 requires a noop_action")

    def reset(self):
        obs, _ = _unpack_reset(self.env.reset())
        return obs

    def step(self, action):
        obs, _, term, trunc, info = _unpack_step(self.env.step(list(action)))
        return obs, term, trunc, info

## The policy contract, as we actually implemented it

All three backbones we ran expose the same two methods, so the loop above
drives them unchanged. Each wrapper is ~150 lines and does nothing but
translate between this contract and the checkpoint's own API.

```python
class <Backbone>Inference:
    def reset(self) -> None: ...
    def step(self, image, instruction, wrist_image=None) -> np.ndarray
        # returns (T, action_dim) in the benchmark's action convention
```

What differs between them is only what comes back:

| backbone | T (actions per call) | views used | notes |
|---|---|---|---|
| OpenVLA | **1** | agent only | one action per forward; no chunk exists |
| UniVLA (Emu3) | ~10 | agent **+ wrist** | raises if the wrist view is missing on a checkpoint trained with it |
| SpatialVLA | chunk | agent only | accepts `wrist_image` and ignores it |

Two things belong in the wrapper and nowhere else, because they are
checkpoint properties rather than method properties:

* **`unnorm_key`** — which dataset's percentile statistics de-normalise the
  action. Passed explicitly and validated against the keys the checkpoint
  actually ships; a wrong key produces plausible-looking but wrong motion.
* **the gripper convention** — the training range and sign differ per
  checkpoint. OpenVLA's LIBERO wrapper rescales `[0,1] -> [-1,1]`,
  binarises by sign, then **inverts**, because LIBERO uses `-1 = open`.
  Doing only one of those two steps gives a policy that reaches correctly
  and never grasps.

Neither belongs in a method notebook — but both must be right before any
intervention result means anything.

## The reference loop

Deliberately plain. The hooks are `image_fn` / `action_fn` parameters that
default to identity, so 02 and 03 are one-line changes rather than forks of
this function.

The one behaviour worth arguing for: **latching `success`**. A benchmark's
termination flag usually means "the goal predicate holds *now*". If the arm
nudges the object afterwards it can flip back, and a solved episode gets
scored as a failure. Latch it and stop the episode there.

In [ ]:
def identity_image(image, state):
    return image


def identity_action(actions, state):
    return actions


def run_episode(adapter, policy, instruction,
                image_fn=identity_image,      # HOOK A
                action_fn=identity_action,    # HOOK B
                state=None):
    """One episode. Returns a dict of per-episode statistics.

    `state` is a free-form dict handed to both hooks, so a hook can keep
    per-episode state (a gaze tracker, a step counter) without this
    function knowing what the hook is.

    The key names below are not cosmetic. They are the names the paired
    tooling reads (`ep_id`, `success`, `steps`, `model_ms_per_infer`,
    `model_ms_per_env_step`), and a record written with other names does
    not load at all rather than loading wrong.
    """
    state = {} if state is None else state
    policy.reset()
    obs = adapter.reset()

    success, done, act_steps = False, False, 0
    model_time, model_calls = 0.0, 0
    info = {}                       # survives an episode with zero steps
    t_start = time.time()

    for _ in range(adapter.settle_steps):      # benchmark-specific; 0 for
        obs, term, trunc, info = adapter.step(adapter.noop_action)
        if term or trunc:                      # most benchmarks
            done = True
            break

    while act_steps < adapter.max_steps and not (success or done):
        image = adapter.get_image(obs)

        # ---- HOOK A: transform the observation ------------------------
        policy_image = image_fn(image, state)

        t0 = time.time()
        actions = policy.step(policy_image, instruction)
        model_time += time.time() - t0
        model_calls += 1

        # ---- HOOK B: transform the actions ----------------------------
        # atleast_2d on both sides: a single-action policy may return a
        # flat (action_dim,) vector, and iterating that would feed the
        # env one scalar per step.
        actions = np.atleast_2d(np.asarray(actions))
        actions = np.atleast_2d(np.asarray(action_fn(actions, state)))

        for row in actions:
            obs, term, trunc, info = adapter.step(row)
            act_steps += 1
            if adapter.is_success(obs, term, info):
                success = True     # latch: see the note above
                break
            if term or trunc:
                done = True
                break
            if act_steps >= adapter.max_steps:
                break

    return {
        "success": bool(success),
        "steps": act_steps,
        "elapsed": time.time() - t_start,
        "model_calls": model_calls,
        "model_ms_per_infer": (model_time / model_calls * 1000) if model_calls else 0.0,
        # Amortised over ENV steps, which is what the robot experiences:
        # with a chunk or a repeat of k, one forward covers k env steps.
        "model_ms_per_env_step": (model_time / max(act_steps, 1)) * 1000,
        # env steps executed per observation -- the quantity action repeat
        # actually changes, and not comparable across backbones unless
        # recorded (see 03).
        "steps_per_call": act_steps / model_calls if model_calls else 0.0,
        # Grasp is not success. Keeping it makes "reached but never held"
        # separable from "held it and dropped it" afterwards; not keeping
        # it means re-running the episode to ask.
        "grasped": adapter.grasped(info),
        "episode_stats": to_jsonable(info.get("episode_stats"))
                         if isinstance(info, dict) else None,
    }

## Adapters — the only part that changes per benchmark

Two examples. Neither is privileged; both are about six lines.

The settle period and no-op action in the LIBERO adapter come from
**OpenVLA's own LIBERO evaluation script**
(`experiments/robot/libero/run_libero_eval.py`). They exist because LIBERO
drops objects onto the table at reset, so the first steps issue a no-op to
let the scene settle. **That is a LIBERO fact, not a general one** — which is
exactly why it lives in an adapter and not in the loop.

When writing a CALVIN adapter, take these from CALVIN's own reference
evaluation. The action layout and gripper sign differ between benchmarks,
and getting the gripper sign wrong yields a policy that reaches correctly
but never grasps — which looks exactly like a method failure.

In [ ]:
def libero_adapter(env, max_steps=220):
    """LIBERO / robosuite: 4-tuple step, success via `done`, settle first."""
    return EnvAdapter(
        env,
        # The 180-degree flip is part of LIBERO's convention, applied by
        # its reference evaluations; it is not a per-policy choice.
        get_image=lambda obs: obs["agentview_image"][::-1, ::-1],
        is_success=lambda obs, term, info: bool(term),
        noop_action=[0, 0, 0, 0, 0, 0, -1],   # -1 = gripper OPEN in LIBERO
        settle_steps=10,
        max_steps=max_steps,
    )


def simpler_env_adapter(env, camera="3rd_view_camera", max_steps=120):
    """SimplerEnv / gymnasium: 5-tuple step, success in info, no settle."""
    return EnvAdapter(
        env,
        get_image=lambda obs: obs["image"][camera]["rgb"],
        is_success=lambda obs, term, info: bool(info.get("success", False)),
        settle_steps=0,
        max_steps=max_steps,
    )

## Running a condition

Every condition must replay the **same initial states**. That is not a
detail: it turns each comparison into matched pairs instead of two
independent samples, and a paired test (McNemar) on the same data is far
more sensitive because episodes where both conditions agree carry no
information about which is better.

### An episode id is not a loop counter

The number that identifies an episode is the number the **environment**
uses to fix the initial state, and it has to be passed to the env's
reset. A bare `for i in range(24)` is not a substitute: it only agrees
with the env's numbering by accident, and where it disagrees the two
conditions are scored on different scenes while still looking paired.

So `run_condition` takes explicit ids per task and refuses to invent
them. The next section gives the ids for both SimplerEnv suites, where
the mapping differs by task family.

In [ ]:
import json
import os


def run_condition(adapter_factory, policy, tasks, condition="baseline",
                  out_dir=None, model_name="", extra=None, **loop_kwargs):
    """Run one condition over `tasks` and return {task: summary}.

    tasks -- {task_name: [ep_id, ...]}. Explicit, because there is no
             safe default: on MoveNear the ids are ordered by object
             triplet, so a prefix such as range(24) is a biased subset
             rather than a smaller unbiased one.

    adapter_factory(task, ep_id) must reset the env to the initial state
    that env calls `ep_id`, and must do so identically in every
    condition. That is the whole basis of the pairing.

    out_dir -- if given, writes
               <out_dir>/<condition>/<task>/results_<task>.json,
               the layout the grid tooling reads.
    extra   -- condition metadata to store at the top of each file, e.g.
               {"action_repeat": 2} or {"depth_prune": 4}.
    """
    if not isinstance(tasks, dict):
        raise TypeError(
            "tasks must be {task_name: [ep_id, ...]}. Passing a bare task "
            "list would mean guessing the episode ids, and a guess that is "
            "wrong produces two conditions that look paired and are not.")

    out = {}
    for task, ep_ids in tasks.items():
        episodes = []
        for ep_id in ep_ids:
            adapter, instruction = adapter_factory(task, ep_id)
            rec = run_episode(adapter, policy, instruction, **loop_kwargs)
            rec.update({"task": task, "ep_id": int(ep_id)})
            episodes.append(rec)
            print(f"  {task} ep {ep_id}: "
                  f"{'SUCCESS' if rec['success'] else 'FAIL':<7} "
                  f"{rec['model_ms_per_infer']:.0f} ms/infer", flush=True)

        n_ok = sum(e["success"] for e in episodes)
        n_grasp = sum(1 for e in episodes if e["grasped"])
        summary = {
            "model": model_name,
            "task": task,             # trusted over the directory name
            "condition": condition,
            "n_episodes": len(episodes),
            "success_rate": n_ok / len(episodes) if episodes else 0.0,
            "grasp_rate": n_grasp / len(episodes) if episodes else 0.0,
            "avg_model_ms_per_infer": float(np.mean(
                [e["model_ms_per_infer"] for e in episodes])),
            "avg_model_ms_per_env_step": float(np.mean(
                [e["model_ms_per_env_step"] for e in episodes])),
            "avg_steps": float(np.mean([e["steps"] for e in episodes])),
            "avg_steps_per_call": float(np.mean(
                [e["steps_per_call"] for e in episodes])),
            # Which GPU this ran on. Latency numbers are not comparable
            # across cards, and a file that does not say which one leaves
            # that as a caveat forever.
            "gpu": _gpu_name(),
            "episodes": episodes,   # per-episode records for paired tests
        }
        summary.update(extra or {})
        print(f"[{task}] {n_ok}/{len(episodes)} = "
              f"{summary['success_rate'] * 100:.1f}%  "
              f"{summary['avg_model_ms_per_infer']:.0f} ms/infer\n")

        if out_dir:
            d = os.path.join(out_dir, condition, task)
            os.makedirs(d, exist_ok=True)
            path = os.path.join(d, f"results_{task}.json")
            with open(path, "w") as fh:
                json.dump(to_jsonable(summary), fh, indent=1)
            print(f"  wrote {path}", flush=True)
        out[task] = summary
    return out


def _gpu_name():
    try:
        import torch
        if torch.cuda.is_available():
            return torch.cuda.get_device_name(0)
    except Exception:
        pass
    return ""

## The episode ids, for both SimplerEnv suites

This is the part that cannot be guessed, and the three Google Robot task
families do it three different ways:

| task family | how an episode index becomes an initial state |
|---|---|
| WidowX-Bridge (4 tasks) | `episode_id`, 0–23 |
| MoveNear | `episode_id`, 0–59 (ordered by object triplet) |
| coke can (3 poses) | **no `episode_id` at all** — a 5 × 5 grid of object init xy over `[−0.35, −0.12] × [−0.02, 0.42]`, indexed by the episode number |

In every case the seed is also set to the episode number. Without it the
URDF variant is drawn from a per-env RNG that is reseeded whenever the
env is rebuilt, so two conditions would be scored on different initial
states — the one failure a paired test cannot detect and cannot survive.

Coke can is the case that matters most here, because it is the one where
a `range(...)` counter silently does something else: it does not reach
the env's placement grid at all.

In [ ]:
# 96 Bridge episodes (4 x 24) and 135 Fractal episodes (3 x 25 + 60).
EPISODE_IDS = {
    # WidowX-Bridge
    "widowx_put_eggplant_in_basket": list(range(24)),
    "widowx_carrot_on_plate":        list(range(24)),
    "widowx_stack_cube":             list(range(24)),
    "widowx_spoon_on_towel":         list(range(24)),
    # Google Robot / Fractal
    "google_robot_pick_horizontal_coke_can": list(range(25)),
    "google_robot_pick_vertical_coke_can":   list(range(25)),
    "google_robot_pick_standing_coke_can":   list(range(25)),
    "google_robot_move_near_v0":             list(range(60)),
}

# "xy_grid" tasks have no episode_id; the index selects a placement.
XY_GRID_TASKS = {
    "google_robot_pick_horizontal_coke_can",
    "google_robot_pick_vertical_coke_can",
    "google_robot_pick_standing_coke_can",
}
XY_GRID = {"x": (-0.35, -0.12, 5), "y": (-0.02, 0.42, 5)}


def reset_options(task, ep_id):
    """-> (seed, options) turning an episode index into ONE fixed state.

    Kept in one function because every condition has to use the same
    mapping; an episode index that means two different things in two
    conditions is not a paired experiment, and nothing downstream can
    detect that it happened.
    """
    ep_id = int(ep_id)
    if task not in XY_GRID_TASKS:
        return ep_id, {"obj_init_options": {"episode_id": ep_id}}
    xs = np.linspace(*XY_GRID["x"])
    ys = np.linspace(*XY_GRID["y"])
    n = len(xs) * len(ys)
    if ep_id >= n:
        raise ValueError(
            f"episode {ep_id} is outside this task's {len(xs)}x{len(ys)} "
            f"placement grid ({n} distinct initial states). Asking for "
            f"more episodes would re-run the same states and inflate n "
            f"without adding information.")
    x, y = xs[ep_id // len(ys)], ys[ep_id % len(ys)]
    return ep_id, {"obj_init_options": {"init_xy": [float(x), float(y)]}}


# Sanity check: the two mappings must not be interchangeable.
assert reset_options("widowx_stack_cube", 7)[1] == {
    "obj_init_options": {"episode_id": 7}}
assert "init_xy" in reset_options(
    "google_robot_pick_standing_coke_can", 7)[1]["obj_init_options"]
assert sum(len(v) for k, v in EPISODE_IDS.items() if k.startswith("widowx")) == 96
assert sum(len(v) for k, v in EPISODE_IDS.items() if k.startswith("google")) == 135
print("episode-id mapping: 96 Bridge + 135 Fractal, coke can on the xy grid")

## Measuring latency without fooling yourself

Two numbers are easy to confuse:

* **ms per model call** — how long one forward costs.
* **ms per environment step** — what the robot actually experiences.

They are only the same when the policy emits one action per call. A policy that
emits a chunk of 10 and executes all of them costs
`model_ms_per_infer / 10` per
environment step. Reporting one as the other makes an already-fast policy look
slow, or vice versa.

Of the three methods here:

| method | ms per call | calls per episode |
|---|---|---|
| foveation | **unchanged** | unchanged |
| action repeat | **unchanged** | **halved** (at repeat=2) |
| depth pruning | **reduced** | unchanged |

So foveation cannot reduce latency at all (the image size, and therefore the
visual token count, is unchanged), action repeat reduces it by making fewer
calls, and depth pruning is the only one that makes a call itself cheaper.

## Portability check — run this before trusting the loop anywhere

Two stub environments differing **only** in which simulator API they speak,
each driven by both a chunked and a single-action policy. All four must
produce the same outcome.

This is the check that catches a loop unpacking a gymnasium 5-tuple into
four names — a `ValueError` on the first step of the first episode, which is
what the earlier version of this notebook did.

In [ ]:
class _StubBase:
    """Reaches its goal after `solve_at` steps. No physics."""

    def __init__(self, solve_at=40, size=64):
        self.solve_at, self.size, self.t = solve_at, size, 0

    def _obs(self):
        rng = np.random.default_rng(self.t)
        return {"cam": rng.integers(0, 256, (self.size, self.size, 3),
                                    dtype=np.uint8)}


class ClassicGymEnv(_StubBase):
    """4-tuple step, reset returns obs, success signalled by `done`."""

    def reset(self):
        self.t = 0
        return self._obs()

    def step(self, action):
        self.t += 1
        return self._obs(), 0.0, self.t >= self.solve_at, {}


class GymnasiumEnv(_StubBase):
    """5-tuple step, reset returns (obs, info), success in info."""

    def reset(self):
        self.t = 0
        return self._obs(), {}

    def step(self, action):
        self.t += 1
        done = self.t >= self.solve_at
        return self._obs(), 0.0, done, False, {"success": done}


class StubPolicy:
    """Emits a chunk of `chunk` actions per call."""

    def __init__(self, chunk=4, action_dim=7):
        self.chunk, self.action_dim = chunk, action_dim

    def reset(self):
        pass

    def step(self, image, instruction):
        return np.zeros((self.chunk, self.action_dim), dtype=np.float32)


class SingleActionPolicy(StubPolicy):
    """Emits ONE action per call, as a flat vector.

    The shape most likely to break a loop that assumes 2-D output."""

    def step(self, image, instruction):
        return np.zeros((self.action_dim,), dtype=np.float32)


get_cam = lambda obs: obs["cam"]

for env_name, cls in [("classic gym", ClassicGymEnv), ("gymnasium", GymnasiumEnv)]:
    for p_name, pol in [("chunk-4", StubPolicy()), ("single", SingleActionPolicy())]:
        adapter = EnvAdapter(cls(), get_image=get_cam, max_steps=200)
        r = run_episode(adapter, pol, "pick up the black bowl")
        print(f"  {env_name:<12} {p_name:<8} success={r['success']} "
              f"steps={r['steps']:>3} calls={r['model_calls']:>3} "
              f"steps/call={r['steps_per_call']:.1f}")
        assert r["success"], f"{env_name}/{p_name} never reached the goal"

# A settle period must not be counted as policy steps.
adapter = EnvAdapter(ClassicGymEnv(solve_at=50), get_image=get_cam,
                     noop_action=[0] * 7, settle_steps=10, max_steps=200)
r = run_episode(adapter, StubPolicy(), "task")
print(f"\n  settle_steps=10, goal at env step 50 -> policy steps={r['steps']}")
assert r["success"] and r["steps"] == 40

# An env that never succeeds must stop at max_steps, not spin forever.
adapter = EnvAdapter(ClassicGymEnv(solve_at=10_000), get_image=get_cam,
                     max_steps=60)
r = run_episode(adapter, StubPolicy(), "task")
print(f"  unsolvable env -> success={r['success']} steps={r['steps']}")
assert not r["success"] and r["steps"] == 60

print("\nloop is portable across both simulator APIs and both policy shapes")

## The output files, checked by round-tripping them

A run that finishes and writes a file the analysis cannot read is the
expensive kind of mistake, so the schema is exercised here on the stub
env rather than discovered after a real campaign.

Two conditions are written to a temporary directory and then paired the
way the analysis pairs them: look up each episode by `ep_id`, keep only
the pairs whose outcome differs, and count them in both directions. The
pairing is a dictionary lookup on `ep_id`, which is why that field's
name and meaning matter more than any other in the file.

In [ ]:
import shutil
import tempfile


class _FlakyEnv(GymnasiumEnv):
    """Succeeds only on the episodes listed, so two conditions differ."""

    def __init__(self, solves):
        super().__init__(solve_at=10 if solves else 10_000)


def _stub_factory(solving_ids):
    def factory(task, ep_id):
        adapter = EnvAdapter(_FlakyEnv(ep_id in solving_ids),
                             get_image=get_cam, max_steps=30)
        return adapter, "pick up the black bowl"
    return factory


tmp = tempfile.mkdtemp()
try:
    plan = {"widowx_stack_cube": list(range(8))}
    run_condition(_stub_factory({0, 1, 2, 3, 4}), StubPolicy(), plan,
                  condition="baseline", out_dir=tmp, model_name="stub")
    run_condition(_stub_factory({0, 1, 5, 6}), StubPolicy(), plan,
                  condition="action_repeat2", out_dir=tmp,
                  model_name="stub", extra={"action_repeat": 2})

    def load(condition, task):
        path = os.path.join(tmp, condition, task, f"results_{task}.json")
        with open(path) as fh:
            summary = json.load(fh)
        # Exactly what the analysis does: index by ep_id, read success.
        return {int(e["ep_id"]): bool(e["success"])
                for e in summary["episodes"]}

    base = load("baseline", "widowx_stack_cube")
    alt = load("action_repeat2", "widowx_stack_cube")
    shared = sorted(set(base) & set(alt))
    fixed = sum(1 for k in shared if alt[k] and not base[k])
    broke = sum(1 for k in shared if base[k] and not alt[k])

    print(f"\n  paired on ep_id: {len(shared)} of 8 episodes")
    print(f"  {fixed} fixed / {broke} broke -> "
          f"delta {100 * (fixed - broke) / len(shared):+.1f} points")
    assert len(shared) == 8, "every episode must pair across conditions"
    assert (fixed, broke) == (2, 3)
finally:
    shutil.rmtree(tmp)

print("\nthe written files pair on ep_id and carry the fields the "
      "analysis reads")

## Porting this to another benchmark (e.g. CALVIN)

**Two** things are benchmark- or backbone-specific. Everything else — the loop,
and all three hooks — stays exactly as it is.

1. **An `EnvAdapter`** for the benchmark: where the camera frame lives, how
   success is reported, whether there is a settle period, and the step cap.
   The gym/gymnasium API difference is already absorbed. CALVIN's success is a
   count of completed subtasks in a sequence rather than a boolean, so decide
   what counts as success for a single episode and put that decision in
   `is_success` — not in the loop.
2. **A policy object** exposing `step(image, instruction) -> (T, action_dim)`
   and `reset()`. A single-action policy may return a flat `(action_dim,)`
   vector; the loop handles both.

One caution when adapting the action convention: gripper sign and
normalisation differ per benchmark **and per checkpoint**, and getting it wrong
produces a policy that reaches correctly but never grasps — which looks like a
method failure rather than a plumbing bug.

### Validate the baseline first

Every result in the grid is a difference against "original policy", so if that
reference is wrong, every other number inherits the error. Before running any
intervention, check the no-intervention condition against the number the
backbone's **own paper** reports for that benchmark.

This is not a formality. Our OpenVLA baseline on `libero_spatial` came out at
74.0% against a published 84.7% — a systematic, reproducible 10.7-point gap
whose cause we still have not identified. Differences measured with the gap
held fixed are still usable, but absolute numbers are not comparable to the
literature until it is understood. Better to find that before the grid than
after.